In [ ]:
import os
import time
import re
import json
import random
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from datetime import datetime
from urllib.parse import quote

try:
    import cloudscraper
    SESSION = cloudscraper.create_scraper(
        browser={"browser": "chrome", "platform": "windows", "mobile": False}
    )
except ImportError:
    import requests
    SESSION = requests.Session()
    SESSION.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8",
    })

BASE = "https://pikabu.ru"
OUTPUT_DIR = "pikabu_brainrot"
os.makedirs(OUTPUT_DIR, exist_ok=True)


SEARCH_QUERIES = [
    "брейнрот", "брейн рот", "итальянский брейнрот",
    "тралалело", "бомбардиро", "балерина капучина",
    "скибиди", "скибиди туалет",
  
    "сигма", "сигма бой", "ризз", "чиназес", "свага",
    "а ниче тот факт", "сидим с бобром", "бобр польский",
    "я в прайме", "6-7", "шесть семь",
  
    "скуф", "нормис", "тюбик", "пикми", "краш",
    "кринж", "имба", "флексить", "дрипчик",
    "ало подростки", "поколение альфа", "поколение зумеры",
    
    "нейросеть мем", "ии мемы", "слоп нейросеть",
    
    "брейнрот", "брейн рот", "brainrot русский", "дегродский контент",
    "тралалело", "тралалеро тралала", "бомбардиро крокодило",
    "балерина капучина", "бобрито бандито", "тунг тунг сахур",
    "итальянский брейнрот", "скибиди туалет", "скибиди",
   
    "а ниче тот факт", "сидим с бобром", "бобр польский",
    "сигма бой", "сигма", "я в прайме", "чиназес",
    "свага", "ало подростки", "шесть семь", "6 7 мем",

    "скуф", "нормис", "тюбик", "пикми", "пик ми герл",
    "редфлаг", "краш", "имба", "кринж", "тильт",
    "флексить", "дрипчик", "анком", "ризз", "ризз русский",
    "фанум такс", "мьюинг", "охайо",

    "нейросеть мем", "ии генерация мем", "слоп",
    
    "skibidi toilet", "sigma male", "rizz", "gyatt", "fanum tax",
    "ohio meme", "mewing", "looksmaxxing", "gigachad", "mogging",
    "npc streamer", "delulu", "goon cave", "baby gronk", "livvy dunne",
    "alpha sigma grindset", "skibidi sigma", "ice spice",

    "gen z humor", "brainrot compilation", "unhinged tiktok",
    "ironic shitpost", "deep fried meme", "absurd humor shorts",

    "subway surfers gameplay reaction", "family guy funny moments tiktok",

    "sigma edit", "phonk edit", "alpha edit", "patrick bateman edit",

    "сигма бой", "скибиди туалет", "ризз русский", "бравл ассасин",
    "оху денно", "тралалело тралала",

    "tralalero tralala", "bombardiro crocodilo", "tung tung sahur",
    "ballerina cappuccina", "italian brainrot",
]

TAGS = [
    "мемы", "юмор", "зумеры", "поколение", "TikTok", "шортсы",
    "ютуб шортс", "скибиди", "сигма", "брейнрот",

    "брейнрот", "брейн рот", "brainrot русский", "дегродский контент",
    "тралалело", "тралалеро тралала", "бомбардиро крокодило",
    "балерина капучина", "бобрито бандито", "тунг тунг сахур",
    "итальянский брейнрот", "скибиди туалет", "скибиди",

    "а ниче тот факт", "сидим с бобром", "бобр польский",
    "сигма бой", "сигма", "я в прайме", "чиназес",
    "свага", "ало подростки", "шесть семь", "6 7 мем",

    "скуф", "нормис", "тюбик", "пикми", "пик ми герл",
    "редфлаг", "краш", "имба", "кринж", "тильт",
    "флексить", "дрипчик", "анком", "ризз", "ризз русский",
    "фанум такс", "мьюинг", "охайо",

    "нейросеть мем", "ии генерация мем", "слоп",

    "skibidi toilet", "sigma male", "rizz", "gyatt", "fanum tax",
    "ohio meme", "mewing", "looksmaxxing", "gigachad", "mogging",
    "npc streamer", "delulu", "goon cave", "baby gronk", "livvy dunne",
    "alpha sigma grindset", "skibidi sigma", "ice spice",

    "gen z humor", "brainrot compilation", "unhinged tiktok",
    "ironic shitpost", "deep fried meme", "absurd humor shorts",

    "subway surfers gameplay reaction", "family guy funny moments tiktok",

    "sigma edit", "phonk edit", "alpha edit", "patrick bateman edit",

    "сигма бой", "скибиди туалет", "ризз русский", "бравл ассасин",
    "оху денно", "тралалело тралала",

    "tralalero tralala", "bombardiro crocodilo", "tung tung sahur",
    "ballerina cappuccina", "italian brainrot",
]


def get_html(url, retries=3):
    """GET с ретраями и случайной задержкой."""
    for attempt in range(retries):
        try:
            r = SESSION.get(url, timeout=20)
            if r.status_code == 200:
                return r.text
            if r.status_code == 429:
                time.sleep(10 + attempt * 5)
                continue
            if r.status_code in (403, 503):
                time.sleep(5)
                continue
        except Exception as e:
            print(f"  [retry {attempt+1}] {e}")
            time.sleep(3)
    return None


def parse_post_card(card):
    """Извлекает базовую инфу из карточки поста на странице листинга."""
    try:
        link_el = card.select_one("a.story__title-link") or card.select_one("a.story-title")
        if not link_el:
            return None
        url = link_el.get("href", "")
        if not url.startswith("http"):
            url = BASE + url

        title = link_el.get_text(strip=True)

        post_id_match = re.search(r"_(\d+)", url)
        post_id = post_id_match.group(1) if post_id_match else None

        rating_el = card.select_one(".story__rating-count, [class*='rating']")
        rating = rating_el.get_text(strip=True) if rating_el else "0"

        author_el = card.select_one(".user__nick, a[class*='user']")
        author = author_el.get_text(strip=True) if author_el else None

        time_el = card.select_one("time")
        published = time_el.get("datetime") if time_el else None

        tag_els = card.select("a.tags__tag")
        tags = [t.get_text(strip=True) for t in tag_els]

        return {
            "post_id": post_id,
            "url": url,
            "title": title,
            "rating": rating,
            "author": author,
            "published": published,
            "tags": ",".join(tags),
        }
    except Exception as e:
        print(f"  [card parse error] {e}")
        return None


def fetch_listing(url, max_pages=5):
    """Универсальный сбор постов с любой страницы листинга (поиск/тег/раздел)."""
    posts = []
    for page in range(1, max_pages + 1):
        sep = "&" if "?" in url else "?"
        page_url = f"{url}{sep}page={page}"
        html = get_html(page_url)
        if not html:
            break

        soup = BeautifulSoup(html, "lxml")
        cards = soup.select("article.story, div.story")

        if not cards:
            break

        for card in cards:
            post = parse_post_card(card)
            if post:
                posts.append(post)

        time.sleep(random.uniform(1.5, 3.0))

    return posts


def fetch_post_comments(post_url):
    """Достаёт пост целиком + все комменты."""
    html = get_html(post_url)
    if not html:
        return None, []

    soup = BeautifulSoup(html, "lxml")

    body_el = soup.select_one(".story__content-inner, .b-story__content")
    body_text = body_el.get_text(" ", strip=True) if body_el else ""

    comments = []

    for script in soup.find_all("script"):
        text = script.string or ""
        if "comments" in text.lower() and "{" in text:
        
            json_matches = re.findall(r"\{[^{}]*\"content\"[^{}]*\}", text)
            for jm in json_matches[:200]:
                try:
                    obj = json.loads(jm)
                    if "content" in obj:
                        comments.append({
                            "comment_id": obj.get("id"),
                            "text": _clean_html(obj.get("content", "")),
                            "author": obj.get("user", {}).get("name") if isinstance(obj.get("user"), dict) else None,
                            "rating": obj.get("rating"),
                            "parent_id": obj.get("parentId"),
                        })
                except Exception:
                    continue
    
    if not comments:
        comment_blocks = soup.select(".comment, [class*='comment__body']")
        for cb in comment_blocks:
            text_el = cb.select_one(".comment__content, [class*='comment-text']")
            author_el = cb.select_one(".user__nick, [class*='user']")
            if text_el:
                comments.append({
                    "comment_id": cb.get("data-id"),
                    "text": text_el.get_text(" ", strip=True),
                    "author": author_el.get_text(strip=True) if author_el else None,
                    "rating": None,
                    "parent_id": None,
                })

    return body_text, comments


def _clean_html(html_text):
    """Чистим HTML-теги из текста коммента."""
    if not html_text:
        return ""
    return BeautifulSoup(html_text, "lxml").get_text(" ", strip=True)


def search_posts(query, max_pages=3):
    """Поиск постов по ключевику."""
    url = f"{BASE}/search.php?q={quote(query)}"
    return fetch_listing(url, max_pages=max_pages)


def tag_posts(tag, max_pages=3):
    """Посты по тегу."""
    url = f"{BASE}/tag/{quote(tag)}/hot"
    return fetch_listing(url, max_pages=max_pages)


def hot_posts(max_pages=3):
    """Горячие посты с главной."""
    return fetch_listing(f"{BASE}/hot", max_pages=max_pages)


def main():
    all_posts = []

    print("=== Поиск по brainrot-сленгу ===")
    for q in tqdm(SEARCH_QUERIES):
        posts = search_posts(q, max_pages=3)
        for p in posts:
            p["source"] = f"search:{q}"
        all_posts.extend(posts)

    print("\n=== Сбор по тегам ===")
    for tag in tqdm(TAGS):
        posts = tag_posts(tag, max_pages=3)
        for p in posts:
            p["source"] = f"tag:{tag}"
        all_posts.extend(posts)

    print("\n=== Горячие посты ===")
    hot = hot_posts(max_pages=5)
    for p in hot:
        p["source"] = "hot"
    all_posts.extend(hot)

    df_posts = pd.DataFrame([p for p in all_posts if p])
    df_posts = df_posts.drop_duplicates(subset=["post_id"]).reset_index(drop=True)
    df_posts.to_csv(f"{OUTPUT_DIR}/posts.csv", index=False)
    print(f"\nУникальных постов: {len(df_posts)}")

    print("\n=== Сбор тел постов и комментариев ===")
    all_comments = []
    bodies = {}

    for i, row in enumerate(tqdm(df_posts.itertuples(), total=len(df_posts))):
        body, comments = fetch_post_comments(row.url)
        bodies[row.post_id] = body

        for c in comments:
            c["post_id"] = row.post_id
            c["post_url"] = row.url
            all_comments.append(c)

        if i > 0 and i % 50 == 0:
            pd.DataFrame(all_comments).to_csv(
                f"{OUTPUT_DIR}/comments_checkpoint.csv", index=False
            )

        time.sleep(random.uniform(1.5, 3.5))  

    df_posts["body"] = df_posts["post_id"].map(bodies)
    df_posts.to_csv(f"{OUTPUT_DIR}/posts_with_body.csv", index=False)

    df_comments = pd.DataFrame(all_comments)
    df_comments = df_comments[
        df_comments["text"].notna() & (df_comments["text"].str.len() > 0)
    ]
    df_comments.to_csv(f"{OUTPUT_DIR}/comments.csv", index=False)
    df_comments.to_parquet(f"{OUTPUT_DIR}/comments.parquet", index=False)

    print(f"\nГотово!")
    print(f"Постов с телом: {len(df_posts)}")
    print(f"Комментариев: {len(df_comments)}")
    print(f"Уникальных авторов: {df_comments['author'].nunique() if 'author' in df_comments else 'n/a'}")


if __name__ == "__main__":
    main()

=== Поиск по brainrot-сленгу ===


100%|██████████| 120/120 [18:48<00:00,  9.40s/it]



=== Сбор по тегам ===


100%|██████████| 96/96 [04:21<00:00,  2.72s/it]



=== Горячие посты ===

Уникальных постов: 387

=== Сбор тел постов и комментариев ===


100%|██████████| 387/387 [19:58<00:00,  3.10s/it]


KeyError: 'text'

In [ ]:
import pandas as pd
df_posts = pd.read_csv("pikabu_brainrot/posts.csv")
print(f"Загружено постов: {len(df_posts)}")
print(df_posts.head())

Загружено постов: 387
    post_id                                                url  \
0  14059469  https://pikabu.ru/story/otvet_na_post_ostalos_...   
1  14059468  https://pikabu.ru/story/_smena_subordinatsionn...   
2  14059467  https://pikabu.ru/story/28_nakhodok_s_aliexpre...   
3  14059466  https://pikabu.ru/story/kamyizyakskiy_tornado_...   
4  14059465  https://pikabu.ru/story/arkticheskie_tanki_kar...   

                                               title  rating        author  \
0  Ответ на пост «Осталось только теорию по билет...     NaN  DeadPuckHome   
1              " Смена субординационного положения "     NaN       Nik1307   
2  28 находок с AliExpress, мимо которых сложно п...     NaN    ElfinSimon   
3                                Камызякский торнадо     NaN       CATAHKA   
4  Арктические «танки» Карельского фронта: как ол...     NaN    NorthKarel   

                   published  \
0  2026-06-13T18:20:03+03:00   
1  2026-06-13T18:19:23+03:00   
2  2026-06-13T18

In [ ]:

test_url = "https://pikabu.ru/community/memes"  

import requests
from bs4 import BeautifulSoup
import re

try:
    import cloudscraper
    SESSION = cloudscraper.create_scraper(
        browser={"browser": "chrome", "platform": "windows", "mobile": False}
    )
except ImportError:
    SESSION = requests.Session()
    SESSION.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "ru-RU,ru;q=0.9",
    })

r = SESSION.get("https://pikabu.ru/hot", timeout=20)
print(f"Статус главной: {r.status_code}")

soup = BeautifulSoup(r.text, "lxml")

story_links = soup.select("a[href*='/story/']")
print(f"Найдено ссылок на story: {len(story_links)}")

if story_links:
    test_url = story_links[0].get("href")
    if not test_url.startswith("http"):
        test_url = "https://pikabu.ru" + test_url
    print(f"Тестовый пост: {test_url}")

    r2 = SESSION.get(test_url, timeout=20)
    print(f"Статус поста: {r2.status_code}, длина HTML: {len(r2.text)}")

    html = r2.text

    print("\n=== Что нашли в HTML ===")
    print(f"Слово 'comment' встречается: {len(re.findall(r'comment', html, re.I))} раз")
    print(f"Класс 'b-comment': {'b-comment' in html}")
    print(f"Класс 'comment__body': {'comment__body' in html}")
    print(f"Класс 'comments__container': {'comments__container' in html}")
    print(f"data-comment-id: {'data-comment-id' in html}")
    print(f"data-story-id: {'data-story-id' in html}")
    print(f"window.gStoryData: {'window.gStoryData' in html or 'storyData' in html}")

    with open("test_post.html", "w", encoding="utf-8") as f:
        f.write(html)
    print(f"\nHTML сохранён в test_post.html — посмотри его")

Статус главной: 200
Найдено ссылок на story: 31
Тестовый пост: https://pikabu.ru/story/esli_pod_vashimi_oknami_dolbit_letom_muzyika__ne_stesnyaytes_kidaytes_yaytsami_14058784
Статус поста: 200, длина HTML: 213497

=== Что нашли в HTML ===
Слово 'comment' встречается: 43 раз
Класс 'b-comment': False
Класс 'comment__body': False
Класс 'comments__container': True
data-comment-id: False
data-story-id: True
window.gStoryData: False

✅ HTML сохранён в test_post.html — посмотри его


In [ ]:

story_id_match = re.search(r"_(\d+)", test_url)
if story_id_match:
    story_id = story_id_match.group(1)
    print(f"Story ID: {story_id}")

    endpoints = [
        ("POST", "https://pikabu.ru/ajax/comments_actions.php",
         {"action": "get_story_comments", "story_id": story_id, "start_comment_id": "0", "limit": "100"}),
        ("GET", f"https://pikabu.ru/story/{story_id}/comments", None),
        ("POST", "https://pikabu.ru/ajax/v1/comments/get",
         {"story_id": story_id, "limit": 100}),
    ]

    for method, url, data in endpoints:
        try:
            if method == "POST":
                r = SESSION.post(url, data=data, timeout=10,
                               headers={"X-Requested-With": "XMLHttpRequest"})
            else:
                r = SESSION.get(url, timeout=10)
            print(f"\n{method} {url}")
            print(f"  Статус: {r.status_code}, длина ответа: {len(r.text)}")
            if r.status_code == 200 and len(r.text) > 50:
                print(f"  Превью: {r.text[:300]}")
        except Exception as e:
            print(f"  Ошибка: {e}")

Story ID: 14058784

POST https://pikabu.ru/ajax/comments_actions.php
  Статус: 200, длина ответа: 2170719
  Превью: {"result":true,"message":"","message_code":0,"data":{"total":1099,"comments":[{"id":395349454,"parent_id":0,"html":"\n\t\t\t\t<div class=\"comment\"  id=\"comment_395349454\" data-id=\"395349454\" data-author-id=\"2926331\" data-author-avatar=\"https://cs14.pikabu.ru/avatars/3661/l3661138-139625232.

GET https://pikabu.ru/story/14058784/comments
  Статус: 404, длина ответа: 1759

POST https://pikabu.ru/ajax/v1/comments/get
  Статус: 404, длина ответа: 16


In [ ]:
import os
import re
import time
import json
import random
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import quote

try:
    import cloudscraper
    SESSION = cloudscraper.create_scraper(
        browser={"browser": "chrome", "platform": "windows", "mobile": False}
    )
except ImportError:
    import requests
    SESSION = requests.Session()
    SESSION.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "ru-RU,ru;q=0.9",
    })

BASE = "https://pikabu.ru"
OUTPUT_DIR = "pikabu_brainrot"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def _clean_html(html_text):
    if not html_text:
        return ""
    return BeautifulSoup(html_text, "lxml").get_text(" ", strip=True)


def fetch_comments_api(story_id, max_total=2000):
    """
    Тянет комменты через ajax/comments_actions.php.
    Эндпоинт отдаёт JSON со всеми комментами (может вернуть до тысяч за раз).
    """
    url = f"{BASE}/ajax/comments_actions.php"
    data = {
        "action": "get_story_comments",
        "story_id": str(story_id),
        "start_comment_id": "0",
        "limit": "100",
        "last_comment_id": "0",
    }

    try:
        r = SESSION.post(
            url, data=data, timeout=30,
            headers={
                "X-Requested-With": "XMLHttpRequest",
                "Referer": f"{BASE}/story/_{story_id}",
            }
        )
        if r.status_code != 200:
            return []

        payload = r.json()
        if not payload.get("result"):
            return []

        raw_comments = payload.get("data", {}).get("comments", [])

        result = []
        for c in raw_comments[:max_total]:
            html_chunk = c.get("html", "")

            sub_soup = BeautifulSoup(html_chunk, "lxml")

            text_el = sub_soup.select_one(".comment__content")
            text = text_el.get_text(" ", strip=True) if text_el else _clean_html(html_chunk)

            author_el = sub_soup.select_one("[data-name], .user__nick, .comment__user")
            author = None
            if author_el:
                author = author_el.get("data-name") or author_el.get_text(strip=True)

            rating_el = sub_soup.select_one(".comment__rating-count, [class*='rating']")
            rating = rating_el.get_text(strip=True) if rating_el else None

            result.append({
                "story_id": story_id,
                "comment_id": c.get("id"),
                "parent_id": c.get("parent_id"),
                "text": text,
                "author": author,
                "rating": rating,
            })

        return result

    except Exception as e:
        print(f"  [api error] story {story_id}: {e}")
        return []


def extract_story_id(url):
    """Достаёт story_id из URL поста."""
    m = re.search(r"_(\d+)", url)
    return m.group(1) if m else None


def get_html(url, retries=3):
    for attempt in range(retries):
        try:
            r = SESSION.get(url, timeout=20)
            if r.status_code == 200:
                return r.text
            if r.status_code == 429:
                time.sleep(10 + attempt * 5)
        except Exception as e:
            time.sleep(3)
    return None


def fetch_post_body(post_url):
    """Тело поста + альтернативный способ узнать story_id (через data-story-id)."""
    html = get_html(post_url)
    if not html:
        return "", None

    soup = BeautifulSoup(html, "lxml")

    body_el = (
        soup.select_one(".story__content-inner") or
        soup.select_one("[class*='story-block']") or
        soup.select_one("article")
    )
    body_text = body_el.get_text(" ", strip=True) if body_el else ""

    story_id = None
    story_el = soup.select_one("[data-story-id]")
    if story_el:
        story_id = story_el.get("data-story-id")

    if not story_id:
        story_id = extract_story_id(post_url)

    return body_text, story_id


posts_path = f"{OUTPUT_DIR}/posts.csv"
if os.path.exists(posts_path):
    df_posts = pd.read_csv(posts_path)
    print(f"Загружено {len(df_posts)} постов из {posts_path}")
else:
    print(f"Файла {posts_path} нет — нужно сначала собрать посты")
    print("Доступные файлы:", os.listdir(OUTPUT_DIR) if os.path.exists(OUTPUT_DIR) else "папки нет")

✅ Загружено 387 постов из pikabu_brainrot/posts.csv


In [ ]:

all_comments = []
bodies = {}
failed_posts = []

for i, row in enumerate(tqdm(df_posts.itertuples(), total=len(df_posts))):
   
    body, story_id = fetch_post_body(row.url)
    bodies[row.url] = body

    if not story_id:
        failed_posts.append(row.url)
        continue

    comments = fetch_comments_api(story_id)

    for c in comments:
        c["post_url"] = row.url
        c["post_title"] = row.title
        all_comments.append(c)

    if i > 0 and i % 30 == 0:
        pd.DataFrame(all_comments).to_csv(
            f"{OUTPUT_DIR}/comments_checkpoint.csv", index=False
        )
        print(f"  [{i}/{len(df_posts)}] комментов собрано: {len(all_comments)}")

    time.sleep(random.uniform(0.8, 1.8))

print(f"\nГотово! Собрано {len(all_comments)} комментариев")
print(f"Постов без story_id: {len(failed_posts)}")

if all_comments:
    df_comments = pd.DataFrame(all_comments)

    df_comments = df_comments[
        df_comments["text"].notna() &
        (df_comments["text"].astype(str).str.len() > 0)
    ]

    df_comments.to_csv(f"{OUTPUT_DIR}/comments.csv", index=False)
    df_comments.to_parquet(f"{OUTPUT_DIR}/comments.parquet", index=False)

    df_posts["body"] = df_posts["url"].map(bodies)
    df_posts.to_csv(f"{OUTPUT_DIR}/posts_with_body.csv", index=False)

    print(f"\nСтатистика:")
    print(f"  Комментариев: {len(df_comments)}")
    print(f"  Средняя длина: {df_comments['text'].str.len().mean():.1f} симв.")
    print(f"  Уникальных авторов: {df_comments['author'].nunique()}")
    print(f"  Уникальных постов: {df_comments['story_id'].nunique()}")

  8%|▊         | 30/387 [01:02<12:43,  2.14s/it]

  [30/387] комментов собрано: 59


 16%|█▌        | 60/387 [02:06<10:57,  2.01s/it]

  [60/387] комментов собрано: 238


 23%|██▎       | 90/387 [03:11<11:33,  2.33s/it]

...

  [330/387] комментов собрано: 3722


 93%|█████████▎| 360/387 [12:53<01:09,  2.59s/it]

  [360/387] комментов собрано: 5447


100%|██████████| 387/387 [13:50<00:00,  2.15s/it]



✅ Готово! Собрано 6914 комментариев
Постов без story_id: 1

📊 Статистика:
  Комментариев: 6632
  Средняя длина: 123.7 симв.
  Уникальных авторов: 5237
  Уникальных постов: 331


In [ ]:
import pandas as pd
import re
from collections import Counter

df = pd.read_csv("pikabu_brainrot/comments.csv")
print(f"Всего комментов: {len(df)}")

BRAINROT_TERMS = [
    "брейнрот", "скуф", "нормис", "сигма", "ризз", "тюбик",
    "пикми", "кринж", "имба", "флекс", "тралалело", "бомбардиро",
    "балерина капучина", "скибиди", "бобр", "ниче тот факт",
    "свага", "прайме", "чиназес", "6-7", "шесть-семь",
    "фанум", "мьюинг", "охайо", "краш", "редфлаг",
]

print("\nЧастоты brainrot-слов:")
for t in BRAINROT_TERMS:
    c = df["text"].str.contains(t, case=False, na=False).sum()
    if c > 0:
        print(f"  {t:25} → {c}")

text = " ".join(df["text"].dropna().astype(str)).lower()
words = re.findall(r"\b[а-яё]{3,}\b", text)
print("\nТоп-50 ру-слов:")
for w, c in Counter(words).most_common(50):
    print(f"  {w}: {c}")

Всего комментов: 6632

Частоты brainrot-слов:
  брейнрот                  → 1
  скуф                      → 24
  нормис                    → 13
  сигма                     → 53
  тюбик                     → 53
  кринж                     → 32
  имба                      → 12
  флекс                     → 2
  скибиди                   → 10
  бобр                      → 5
  6-7                       → 4
  краш                      → 17

Топ-50 ру-слов:
  что: 1986
  это: 1452
  как: 1081
  так: 810
  все: 622
  если: 559
  вот: 446
  для: 441
  уже: 424
  или: 400
  только: 396
  там: 357
  просто: 351
  было: 345
  меня: 334
  есть: 331
  надо: 325
  они: 319
  можно: 315
  когда: 314
  мне: 312
  кто: 307
  нет: 305
  его: 299
  лет: 289
  ещё: 273
  всё: 273
  тоже: 269
  даже: 262
  будет: 252
  без: 252
  чтобы: 235
  вообще: 235
  тут: 231
  где: 223
  может: 218
  она: 216
  еще: 215
  чем: 206
  сейчас: 203
  про: 203
  при: 199
  был: 187
  потом: 180
  нас: 179
  какой: 174
  р